# NASA C-MAPSS: simulated trajectories and RUL alignment

Evidence class: simulated engine run-to-failure trajectories and held-out remaining-useful-life (RUL) files; this is not real-fleet evidence. The notebook verifies each FD001-FD004 file triplet separately and charts only report-derived row/unit and RUL-alignment values. Subsets are never pooled.

Expected local layout: `data/cmapss/{train,test,RUL}_FD00N.txt`, or set `ISOPRAX_DATA_DIR` before starting Jupyter.

In [ ]:
import os
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
working_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file() and (candidate / "isoprax").is_dir()
    ),
    None,
)
configured_root = os.environ.get("ISOPRAX_REPO_ROOT")
REPO_ROOT = (
    Path(configured_root).expanduser().resolve() if configured_root else working_root
)
if (
    REPO_ROOT is None
    or not (REPO_ROOT / "pyproject.toml").is_file()
    or not (REPO_ROOT / "isoprax").is_dir()
):
    raise RuntimeError(
        "Launch from this checkout or set ISOPRAX_REPO_ROOT to its repository root."
    )
if configured_root and working_root is not None and REPO_ROOT != working_root:
    raise RuntimeError(
        "ISOPRAX_REPO_ROOT does not match the notebook's checkout directory."
    )
repo_path = str(REPO_ROOT)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

import isoprax

PACKAGE_ROOT = Path(isoprax.__file__).resolve().parents[1]
if PACKAGE_ROOT != REPO_ROOT:
    raise RuntimeError(
        "The selected Python kernel does not import Isoprax from this checkout."
    )

from IPython.display import Markdown, display

from notebooks._support import (
    data_root_from_environment,
    display_review,
    review_dataset,
)

DATA_ROOT = data_root_from_environment(REPO_ROOT)
CMAPSS_ROOT = DATA_ROOT / "cmapss"
print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Local C-MAPSS directory: {CMAPSS_ROOT}")

In [ ]:
CMAPSS_REVIEWS = {}
for subset in ("FD001", "FD002", "FD003", "FD004"):
    display(Markdown(f"## {subset} — verified independently"))
    review = review_dataset(
        "cmapss",
        {
            "dataset": subset,
            "train": CMAPSS_ROOT / f"train_{subset}.txt",
            "test": CMAPSS_ROOT / f"test_{subset}.txt",
            "rul": CMAPSS_ROOT / f"RUL_{subset}.txt",
        },
        REPO_ROOT,
    )
    CMAPSS_REVIEWS[subset] = review
    display_review(review)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import HistGradientBoostingRegressor

from notebooks import _analysis

for subset in ("FD001", "FD002", "FD003", "FD004"):
    subset_review = CMAPSS_REVIEWS[subset]
    if subset_review.verification.status not in {"verified", "verified_with_warnings"}:
        display(
            Markdown(
                f"## {subset}: analysis not run because verification did not pass."
            )
        )
        continue

    display(Markdown(f"## {subset}: independent trajectory and RUL analysis"))
    train_path = CMAPSS_ROOT / f"train_{subset}.txt"
    test_path = CMAPSS_ROOT / f"test_{subset}.txt"
    rul_path = CMAPSS_ROOT / f"RUL_{subset}.txt"
    train = pd.read_csv(
        train_path, sep=r"\s+", header=None, names=_analysis.CMAPSS_COLUMNS, engine="c"
    )
    test = pd.read_csv(
        test_path, sep=r"\s+", header=None, names=_analysis.CMAPSS_COLUMNS, engine="c"
    )
    train_rul = _analysis.derive_cmapss_training_rul(
        train["unit_number"], train["cycle"]
    )
    final_rows, actual_rul = _analysis.align_cmapss_test_rul(
        test["unit_number"], test["cycle"], np.loadtxt(rul_path, dtype=float, ndmin=1)
    )

    train_life = train.groupby("unit_number")["cycle"].max()
    print(
        f"Train units: {train_life.size}; test units: {len(final_rows)}; train rows: {len(train):,}; test rows: {len(test):,}"
    )
    display(train_life.describe().to_frame("training cycles per run-to-failure engine"))
    example_unit = train["unit_number"].iloc[0]
    example = train.loc[train["unit_number"].eq(example_unit)]
    figure, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    train_life.plot.hist(bins=20, ax=axes[0], color="#3977a8")
    axes[0].set_title(f"{subset}: training engine life")
    pd.Series(train_rul).plot.hist(bins=20, ax=axes[1], color="#b87932")
    axes[1].axvline(np.median(train_rul), color="black", linestyle=":", label="median")
    axes[1].set_title(f"{subset}: train-derived row RUL")
    axes[1].set_xlabel("Remaining cycles within training run")
    axes[1].legend()
    for sensor in ("sensor_2", "sensor_7"):
        axes[2].plot(example["cycle"], example[sensor], label=sensor)
    axes[2].set_title(f"{subset}: example training engine {example_unit}")
    axes[2].set_xlabel("Cycle")
    axes[2].legend()
    display(figure)
    plt.close(figure)

    X_train = train.loc[:, _analysis.CMAPSS_MODEL_FEATURES]
    X_test = test.iloc[final_rows].loc[:, _analysis.CMAPSS_MODEL_FEATURES]
    median_prediction = np.full(len(actual_rul), float(np.median(train_rul)))
    model = HistGradientBoostingRegressor(
        max_iter=100, l2_regularization=1.0, random_state=41
    )
    model.fit(X_train, train_rul)
    model_prediction = model.predict(X_test)
    result_rows = [
        {
            "model": "training-label median",
            **_analysis.regression_metrics(actual_rul, median_prediction),
        },
        {
            "model": "histogram gradient boosting",
            **_analysis.regression_metrics(actual_rul, model_prediction),
        },
    ]
    display(pd.DataFrame(result_rows).set_index("model"))
    print(
        "Test evaluation uses only each official test unit’s final observed row and its aligned official RUL target."
    )

    figure, axis = plt.subplots(figsize=(5, 5), constrained_layout=True)
    axis.scatter(actual_rul, model_prediction, alpha=0.65, label="model")
    low = min(float(np.min(actual_rul)), float(np.min(model_prediction)))
    high = max(float(np.max(actual_rul)), float(np.max(model_prediction)))
    axis.plot(
        [low, high],
        [low, high],
        linestyle=":",
        color="black",
        label="perfect agreement",
    )
    axis.set_title(f"{subset}: official test RUL vs prediction")
    axis.set_xlabel("Official RUL (cycles)")
    axis.set_ylabel("Predicted RUL (cycles)")
    axis.legend()
    display(figure)
    plt.close(figure)

Each RUL file must align one-to-one with the matching test units. Training RUL is derived within each run-to-failure engine. Per subset, the fixed model is compared with a training-median predictor on only the final observed row of each official test unit and its aligned RUL target. Unit IDs are not predictors; test targets are not used for tuning; FD001–FD004 are never pooled.